# 3-Qubit Chip

A Step-by-Step Superconducting Quantum Chip Design Tutorial: from Theory to Simulation

Tutorial Videos: https://www.youtube.com/playlist?list=PLnK6MrIqGXsJF6XLP-1jIBBhsCS5ZQSmR

James Saslow, Shreyan Juvvadi, Hiu Yung Wong*

contact: Hiu-Yung Wong, hiuyung.wong@sjsu.edu

In [1]:
# NOTE: Only run this cell if 'qiskit_metal' package is outside the folder where chip_version_number.ipynb is located

import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..'))) # Allows access to qiskit_metal folder outside this folder

In [2]:
%load_ext autoreload
%autoreload 2

# Importing Packages
import numpy as np
from pandas import DataFrame


import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs

from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket 
from qiskit_metal.qlibrary.qubits.transmon_cross_fl import TransmonCrossFL
from qiskit_metal.qlibrary.qubits.transmon_pocket_teeth import TransmonPocketTeeth


from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight

from qiskit_metal.qlibrary.lumped.cap_n_interdigital import CapNInterdigital
from qiskit_metal.qlibrary.couplers.cap_n_interdigital_tee import CapNInterdigitalTee

from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from qiskit_metal.qlibrary.couplers.line_tee import LineTee

from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround


from qiskit_metal.analyses.simulation.scattering_impedance import ScatteringImpedanceSim
from qiskit_metal.analyses.quantization import LOManalysis
from qiskit_metal.renderers.renderer_ansys.ansys_renderer import QAnsysRenderer



08:15AM 18s WARNING [_qt_message_handler]: WARNING: monitorData: Unable to obtain handle for monitor '\\.\DISPLAY1', defaulting to 96 DPI. (No context available from Qt)
Python Traceback (most recent call last):
  File "C:\Users\012738063\AppData\Local\anaconda3\envs\qmetal3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\012738063\AppData\Local\anaconda3\envs\qmetal3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\012738063\AppData\Local\anaconda3\envs\qmetal3\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "C:\Users\012738063\AppData\Local\anaconda3\envs\qmetal3\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "C:\Users\012738063\AppData\Local\anaconda3\envs\qmetal3\Lib\asyncio\base_events.py", line 645, in run_forever
    self._run_once()
  File "C:\U

In [3]:
# NOTE: Custom Qiskit Metal Qubit Imports
from qiskit_metal.qlibrary.user_components.XYZ import XYZ            # Custom Qubit
from qiskit_metal.qlibrary.user_components.L_turn import L_turn    # L-Bracket Wire

In [4]:
# Determining Resonator Parameters


def get_length(target_freq, epsilon_eff):
    '''
    Determining the length of the resonator
    '''

    # Speed of Light
    c = 299_792_458 #m/s

    Lambda = c / (target_freq * np.sqrt(epsilon_eff)) * 1000 # In mm 

    return Lambda # Returns the wavelength in millimeters

target_freq4 = 6.86 * 10**9 #Hz --- for Q0_bottom

target_freq2 = 7.6 * 10**9 #Hz --- for Q0_top
target_freq1 = 7.4 * 10**9 #Hz --- for Q1_top

epsilon_eff = 6.1 #5.22


L1 = np.round(get_length(target_freq1, epsilon_eff)/4 , 2)
L2 = np.round(get_length(target_freq2, epsilon_eff)/4 , 2)
L4 = np.round(get_length(target_freq4, epsilon_eff)/4 , 2)

L2_coupled = np.round(get_length(5.3*10**9, epsilon_eff)/2 , 2)




L4 = np.round(get_length(target_freq4, epsilon_eff)/4 , 2)

print('L1 = ', L1, 'mm')
print('L2 = ', L2, 'mm')
print('L4 = ', L4, 'mm')
print('L2_coupled = ', L2_coupled, 'mm')


# Determining Coplanar Waveguide Parameters

# ==== The Following Parameters yield an impedence of 50 ohms ====

cpw_width = 10.0 /1000  #mm # Coplanar waveguide width (The width of the blue wire)
cpw_gap   = 5.806 /1000 #mm # Ground Plane Spacing (Space between edge of blue wire and ground plane)

cpw_width_bias = 12.00 / 1000 #mm # For flux bias current
cpw_gap_bias   = 6.959 / 1000 #mm # For flux bias current






L1 =  4.1 mm
L2 =  3.99 mm
L4 =  4.42 mm
L2_coupled =  11.45 mm


In [5]:
# Prompting GUI

design = designs.DesignPlanar({}, True)
design.overwrite_enabled = True # Editing Enabled

# Constraining Chip Size
design.chips.main.size['size_x'] = '5mm'
design.chips.main.size['size_y'] = '5mm'

gui = MetalGUI(design)



# Variables

In [6]:
# Quantum Chip Variables (User Input)


# ============================================================
# Chip Dimensions

margin_x = 0.46 # mm
margin_y = 0.35 # mm


vertical_lp_spacing = 2.20 #mm
bottom_lp_spacing   = 1.68 #mm
top_lp_spacing      = 3.76 #mm

# ============================================================
# Transmission Line

# coupling_length = 0.2    # mm
hanger_separation = 0.85 # mm


# ============================================================
# Bottom Qubit Placement

bottom_vert_displacement = 0.59 # m   Displacement from Bottom of the chip to the bottom of JJ(s)



# ============================================================
# Bottom Qubit 1

qubit1_bottom_shift = 0.5 #


# ============================================================
# Top Qubit Displacement
top_vert_displacement = 1.25 + 1 # mm
top_separation = 2 # mm



# ============================================================
# L Bracket

L_qubit_gap = 0.005 # mm   Gap between the gray spaces of the transmon pocket and the gray region of the L-Bracket



# ============================================================
# HFSS or GDS

GDS = False # Set (GDS = True) to export to GDS and (GDS = False) for HFSS analysis



# =============================================================
# Resonators

qubit_pad_width_b0 = 0.4 # mm
qubit_pocket_width_b0 = 0.650 # mm
mini_cap_length_b0 = 0.125 # mm -- pad width 'b'
outer_length_b0 = 0.1 # mm
coupling_length_b0 = 0.2+0.05    # mm
hanger_separation_b0 = 0.85 # mm


prong_length_b0 = outer_length_b0 + 0.5 * (qubit_pocket_width_b0 - qubit_pad_width_b0) + mini_cap_length_b0 # Total Length of the Capacitor Qubit Prong


L4_eff = L4 - prong_length_b0 - coupling_length_b0


print('L4_eff = ', L4_eff, 'mm')


qubit_pad_width_t1 = 0.4 # mm
qubit_pocket_width_t1 = 0.650 # mm
mini_cap_length_t1 = 0.125 # mm -- pad width 'b'
outer_length_t1 = 0.1 # mm
coupling_length_t1 = 0.2+0.05    # mm
hanger_separation_t1 = 0.85 # mm




qubit_pad_width_t0 = 0.4 # mm
qubit_pocket_width_t0 = 0.650 # mm
mini_cap_length_t0 = 0.125 # mm -- pad width 'b'
outer_length_t0 = 0.1 # mm
coupling_length_t0 = 0.2+0.05    # mm
hanger_separation_t0 = 0.85 # mm

prong_length_t1 = outer_length_t1 + 0.5 * (qubit_pocket_width_t1 - qubit_pad_width_t1) + mini_cap_length_t1 # Total Length of the Capacitor Qubit Prong
prong_length_t0 = outer_length_t0 + 0.5 * (qubit_pocket_width_t0 - qubit_pad_width_t0) + mini_cap_length_t0 # Total Length of the Capacitor Qubit Prong

L1_eff = L1 - prong_length_t1 - coupling_length_t1
L2_eff = L2 - prong_length_t0 - coupling_length_t0
# =============================================================
L2_coupled_eff = L2_coupled - prong_length_t1 - prong_length_t0

print('L1_eff = ', L1_eff, 'mm')
print('L2_eff = ', L2_eff, 'mm')


L4_eff =  3.8200000000000003 mm
L1_eff =  3.4999999999999996 mm
L2_eff =  3.39 mm


# Launch Pads

In [7]:
transmission_lpL = LaunchpadWirebond(design, 'Transmission_Launch_Pad_L',
                                options = dict(pos_x = str(-2.5 + margin_x  ) + 'mm', 
                                                      pos_y = str(-vertical_lp_spacing/2) + 'mm', 
                                                      orientation = '0',
                                                      trace_width = cpw_width,
                                                      trace_gap = cpw_gap))


transmission_lpR = LaunchpadWirebond(design, 'Transmission_Launch_Pad_R',
                                options = dict(pos_x = str(2.5 - margin_x  ) + 'mm', 
                                                      pos_y = str(-vertical_lp_spacing/2) + 'mm', 
                                                      orientation = '180',
                                                      trace_width = cpw_width,
                                                      trace_gap = cpw_gap))

lp0_top_control = LaunchpadWirebond(design, 'Qubit_0_Top_Control_Launch_Pad',
                                options = dict(pos_x = str(-2.5 + margin_x  ) + 'mm', 
                                                      pos_y = str(vertical_lp_spacing/2) + 'mm', 
                                                      orientation = '0',
                                                      trace_width = cpw_width_bias,
                                                      trace_gap = cpw_gap))



lp1_top_control = LaunchpadWirebond(design, 'Qubit_1_Top_Control_Launch_Pad',
                                options = dict(pos_x = str(2.5 - margin_x  ) + 'mm', 
                                                      pos_y = str(vertical_lp_spacing/2) + 'mm', 
                                                      orientation = '180',
                                                      trace_width = cpw_width_bias,
                                                      trace_gap = cpw_gap))


gui.rebuild()
gui.autoscale()

# Transmission Line

In [8]:
# Making Hangers

def make_hanger(name, pos_x, pos_y,coupling_length,flop = False, mirror = False):
    
    orientation = '180' if flop else '0'
    return CoupledLineTee(design, name , options=dict(pos_x= pos_x,
                                             pos_y= pos_y,
                                             coupling_length=coupling_length,
                                             prime_gap = cpw_gap,
                                             prime_width = cpw_width,
                                             second_gap = cpw_gap,
                                             second_width = cpw_width,
                                             orientation = orientation,
                                             mirror = mirror,
                                             open_termination = False)) # DEFAULT: FALSE





y_trans = str(-vertical_lp_spacing/2) + 'mm' # y-position of the transmission line

TQ1 = make_hanger('TQ1', str(-0.5*hanger_separation_b0 - 0.5*coupling_length_b0+0.3)+'mm', y_trans,coupling_length_b0, mirror = False)
TQ0 = make_hanger('TQ0', str(-1.5*hanger_separation_b0 - 1.5*coupling_length_b0)+'mm', y_trans,coupling_length_b0 ,mirror = True, flop = True) # b0
TQ3 = make_hanger('TQ3', str(1.5*hanger_separation_t1  + 1.5*coupling_length_t1) + 'mm', y_trans,coupling_length_t1 ,flop = True)

gui.rebuild()
gui.autoscale()


In [9]:
# Making Routes Between Hangers


def make_cpw(comp1, pin1, comp2, pin2, name):
    ops = Dict(hfss_wire_bonds = False, # Set 'True' for air bridges
               trace_width = cpw_width,
               trace_gap = cpw_gap,
              pin_inputs=Dict(
                 start_pin=Dict(
                     component=comp1,
                     pin= pin1),
                 end_pin=Dict(
                     component=comp2,
                     pin=pin2)))
    
    return RouteStraight(design, name, options=ops)


cpw_left  = make_cpw('Transmission_Launch_Pad_R', 'tie', 'TQ1', 'prime_end', 'cpw_left')
cpw_right = make_cpw('TQ1', 'prime_start', 'Transmission_Launch_Pad_L', 'tie', 'cpw_right')


gui.rebuild()
gui.autoscale()


# Bottom Qubit

In [10]:
def num(string):
    # 'mm' or 'um' are accepted units
    
    unit = string[-2:]
    number = float(string[:-2])

    if unit == 'mm':
        return number
    elif unit == 'um':
        return number / 1000 # Converting um to mm
    else:
        raise ValueError




def make_qubit(features, qubit_name, L_name):

    # ==================== Adding mini-capacitors ==========================
    # Just including 'b' capacitor pad
    if (features["include_a"] == False) and (features["include_b"] == True):
        connection_pads=dict( 
            b = dict(loc_W=-1,loc_H=+1,
                    pad_gap   = features["mini_gap"],
                    cpw_width = features["cpw_width"],
                    cpw_gap   = features["cpw_gap"],
                    pocket_rise = '0um',
                    pad_height = features["cpw_width"],
                    pad_cpw_shift = 0,
                    pad_width = features["mini_cap_lengthb"],
                    cpw_extend = features["outer_length"])
        )
    
    elif (features["include_a"] == True) and (features["include_b"] == False):
        connection_pads=dict( 
            b = dict(loc_W=+1,loc_H=+1,
                    pad_gap   = features["mini_gap"],
                    cpw_width = features["cpw_width"],
                    cpw_gap   = features["cpw_gap"],
                    pocket_rise = '0um',
                    pad_height = features["cpw_width"],
                    pad_cpw_shift = 0,
                    pad_width = features["mini_cap_length"],
                    cpw_extend = features["outer_length"])
        )


    # Including both 'a' and 'b' capacitor pads
    elif (features["include_a"] == True) and (features["include_b"] == True):
        connection_pads=dict( 
            a = dict(loc_W=+1,loc_H=+1,
                    pad_gap   = features["mini_gap"],
                    cpw_width = features["cpw_width"],
                    cpw_gap   = features["cpw_gap"],
                    pocket_rise = '0um',
                    pad_height = features["cpw_width"],
                    pad_cpw_shift = 0,
                    pad_width = features["mini_cap_length"],
                    cpw_extend = features["outer_length"]),
            b = dict(loc_W=-1,loc_H=+1,
                    pad_gap   = features["mini_gap"],
                    cpw_width = features["cpw_width"],
                    cpw_gap   = features["cpw_gap"],
                    pocket_rise = '0um',
                    pad_height = features["cpw_width"],
                    pad_cpw_shift = 0,
                    pad_width = features["mini_cap_lengthb"],
                    cpw_extend = features["outer_length"])
        )


    options = dict(
        pad_width     =  features["pad_width"], 
        pad_height    =  features["pad_height"],
        pocket_height =  features["pocket_height"],
        pocket_width  =  features["pocket_width"],
        
        # Adding mini-capacitors
        connection_pads = connection_pads        
        )
    
    qubit = XYZ(design, qubit_name, options = dict(
        pos_x = features["pos_x"],
        pos_y = features["pos_y"],
        cpw_gap = features["cpw_gap"] ,
        cpw_width = features["cpw_width"],
        pad_gap = features["big_gap"], 
        orientation = features["orientation"],
        GDS = features["GDS"],
        double = features["double"],
        dx = features["dx"],
        **options))
    

    if features["include_L"] == True:

        pos_x = str( num(features["pos_x"])+num(features["L_offset"])) + 'mm'
       

        orientation = '0'
        pos_y = str(- num(features["L_spacing"]) + num(features["pos_y"]) - num(features["big_gap"])/2 ) +'mm'
        
        if features["flip"] == True:
            orientation = '180'
            pos_y = str(num(features["L_spacing"]) + num(features["pos_y"]) + num(features["big_gap"])/2 ) +'mm'   
            if features["mirror"] == True:
                pos_x = str( num(features["pos_x"])+num(features["L_offset"]) ) + 'mm'
        elif (features["flip"] == False) and (features["mirror"] == False):
            pos_x = str( num(features["pos_x"]) - num(features["L_Turn_length"])/2+num(features["L_offset"])) + 'mm'



        L_bracket = L_turn(design, L_name, options = dict(pos_x = pos_x,
                                                    pos_y = pos_y, # 5um Spacing
                                                    coupling_length = features["L_Turn_length"],
                                                    mirror = features["mirror"], # Mirrors around y-axis
                                                    second_width =  features["cpw_width_bias"],
                                                    second_gap   =  features["cpw_gap_bias"],
                                                    orientation = orientation,
                                                    fillet = features["fillet"]
                                                    ))
        
        return qubit, L_bracket # Returns both qubit and L_bracket object
    
    else:
        return qubit # Only Returns the Qubit


In [11]:
# Bottom 0 Qubit

b0_features = {

    # Position
    "pos_x"         : str( -bottom_lp_spacing/2 - qubit1_bottom_shift+0.39-0.7) + 'mm',             # x - position
    "pos_y"         : str(-2.5 + bottom_vert_displacement + 0.15) + 'mm',   # y - position
    "orientation"   : "0",                                           # Angular Position

    # Qubit Features
    "pad_width"      : str(qubit_pad_width_b0) + 'mm',
    "pad_height"     : "0.1mm",
    "pocket_width"   : str(qubit_pocket_width_b0) + 'mm',       # Width of the gray space
    "pocket_height"  : "325um",         # Height of the gray space
    #test123 "big_gap"        : "40um",          # Separation space between the big capacitor and the edge of the pocket
    "big_gap"        : "60um",          # Separation space between the big capacitor and the edge of the pocket


    # Capacitor Features
    "include_a"      : True,           # Set True to include the leftmost mini-capacitor
    "include_b"      : False,            # Set True to include the rightmost mini-capacitor
    #test123 "mini_gap"       : "45um",          # Separation space between the mini capacitor and the big capacitor
    "mini_gap"       : "45um",          # Separation space between the mini capacitor and the big capacitor

    "mini_cap_length": mini_cap_length_b0,       # Length of the mini-capacitor
    "mini_cap_lengthb": mini_cap_length_b0,       # Length of the mini-capacitor    
    "outer_length"   : outer_length_b0,

    # Wire Features
    "cpw_width"      : cpw_width,       # Blue space of the wire
    "cpw_gap"        : cpw_gap,         # Gray space of the wire

    # Junctions
    "double"         : True,            # True for 2 x JJ's, False for 1 x JJ 
    "dx"             :"10um",           # 2*dx = separation distance of JJ's (Can be any numerical value if double = False)

    # GDS Export Option
    "GDS"            : False,           # True for GDS export, False for HFSS export

    # L_Turn Design
    "include_L"      : True,           # Set True to include the L_Turn, set False to remove it
    "L_offset"       : "100um", 
    "L_Turn_length"  : "120um",         # Length of the L Turn
    "L_spacing"      : "15um",           # Distance between the bottom of the qubit and the top of the L - wire
    "cpw_width_bias" : cpw_width_bias,  # Blue space of the bias wire
    "cpw_gap_bias"   : cpw_gap_bias,    # Gray space of the bias wire
    "flip"           : False,           # Flips (re-orients) the L-Turn 180 degrees
    "mirror"         : False,           # Mirrors the L-Turn
    "fillet"         : '99um'           # Fillet of L-turn (Radius Curvature)  -- default setting: 20um
}



# b0, L_turn_b0 = make_qubit(b0_features, 'Q0_bottom', 'L_bracket0_bottom')
b0  = make_qubit(b0_features, 'Q0_bottom', 'L_brackeb0_bottom')

gui.rebuild()
gui.autoscale()

# Meander

In [12]:
# Meander for Qubit 0


def make_meander(comp1, pin1, comp2, pin2, length, name, spacing = '80um', asymmetry = '0um', fillet = '50um'):
    ops=dict(fillet= fillet)

    options = Dict(
        total_length= str(length) + 'mm',
        hfss_wire_bonds = False,
        trace_width = cpw_width,
        trace_gap   = cpw_gap,

        meander = Dict(
            spacing = spacing,
            asymmetry = asymmetry
        ),

        pin_inputs=Dict(
            start_pin=Dict(
                component=comp1,
                pin= pin1),
            end_pin=Dict(
                component= comp2,
                pin=pin2)),
        lead=Dict(
            start_straight= '0.10mm',
            end_straight = '0.45mm'
            ),
        **ops
    )

    return RouteMeander(design, name, options=options)



Q0_bottom_meander = make_meander('TQ1', 'second_end', 'Q0_bottom', 'b', L4_eff, 'meander_Q0_bottom', spacing = '90um', asymmetry= '100um',fillet = '40um')



gui.rebuild()
gui.autoscale()


# Flux Bias

In [13]:


# Double Qubit Control Mechanism 


lp0_bottom = LaunchpadWirebond(design, 'Launch_Pad_0_Bottom',
                                options = dict(pos_x = str( -bottom_lp_spacing/2  ) + 'mm', 
                                                      pos_y = str(-2.5 + margin_y) + 'mm', 
                                                      orientation = '90',
                                                      trace_width = cpw_width_bias,
                                                      trace_gap = cpw_gap_bias))





gui.rebuild()
gui.autoscale()

In [16]:



def make_bias0(launchpad, launchpad_pin, L_bracket, L_bracket_pin, name, start_straight = '0.1um', end_straight = '0.1um'):
    pathfinder_options= dict(
        pin_inputs=dict(
            start_pin=dict(component=launchpad, pin=launchpad_pin),
            end_pin=dict(component=L_bracket, pin= L_bracket_pin)
        ),
        trace_width=cpw_width_bias,
        trace_gap = cpw_gap_bias,
        fillet='99um',
        # lead=dict(start_straight= start_straight, 
        #           end_straight= end_straight)
                  )

    return RouteStraight(design, name, options=pathfinder_options)



def make_bias1(launchpad, launchpad_pin, L_bracket, L_bracket_pin, name, start_straight = '0.1um', end_straight = '0.1um'):
    pathfinder_options= dict(
        pin_inputs=dict(
            start_pin=dict(component=launchpad, pin=launchpad_pin),
            end_pin=dict(component=L_bracket, pin= L_bracket_pin)
        ),
        trace_width=cpw_width_bias,
        trace_gap = cpw_gap_bias,
        fillet='99um',
        lead=dict(start_straight= start_straight, 
                  end_straight= end_straight)
                  )

    return RoutePathfinder(design, name, options=pathfinder_options)


# bias0_bottom = make_bias1('Launch_Pad_0_Bottom','tie', 'L_bracket0_bottom', 'second_end', 'bias0_bottom')


# gui.rebuild()
# gui.autoscale()

In [17]:

pathfinder_options= dict(
    pin_inputs=dict(
        start_pin=dict(component= 'Launch_Pad_0_Bottom', pin = 'tie'),
        end_pin=dict(component= 'stg1', pin= 'short')
    ),
    trace_width=cpw_width_bias,
    trace_gap = cpw_gap_bias,
    fillet='99um',
    lead=dict(start_straight= '10um', 
                end_straight= '10um')
                )
#old method
#rpf = RoutePathfinder(design, 'bias0_bottom', options=pathfinder_options)
rpf = make_bias1('Launch_Pad_0_Bottom','tie', 'L_brackeb0_bottom', 'second_end', 'bias0_bottom', start_straight='10um', end_straight='100um')


gui.rebuild()
gui.autoscale()



# Top Qubit

In [18]:


# Top 0 Qubit

t0_features = {
    # Position
    "pos_x"         : str(-top_separation/2) + 'mm',               # x - position
    "pos_y"         : str(2.5 - top_vert_displacement ) + 'mm',   # y - position
    "orientation"   : "180",                                           # Angular Position

    # Qubit Features
    "pad_width"      : str(qubit_pad_width_t0) + 'mm',
    "pad_height"     : "0.1mm",
    "pocket_width"   : str(qubit_pocket_width_t0) + 'mm',       # Width of the gray space
    "pocket_height"  : "325um",         # Height of the gray space
    "big_gap"        : "60um",          # Separation space between the big capacitor and the edge of the pocket

    # Capacitor Features
    "include_a"      : True,           # Set True to include the rightmost mini-capacitor
    "include_b"      : True,            # Set True to include the leftmost mini-capacitor
    "mini_gap"       : "45um",          # Separation space between the mini capacitor and the big capacitor
    "mini_gap2"       : "45um",          # Separation space between the mini capacitor and the big capacitor    
    "mini_cap_length": str(mini_cap_length_t0) + 'mm',       # Length of the mini capacitor
    "mini_cap_lengthb": str(0.001) + 'mm',       # Length of the mini capacitor    
    "outer_length"   : str(outer_length_t0) + 'mm',

    # Wire Features
    "cpw_width"      : cpw_width,       # Blue space of the wire
    "cpw_gap"        : cpw_gap,         # Gray space of the wire

    # Junctions
    "double"         : True,            # True for 2 x JJ's, False for 1 x JJ 
    "dx"             :"10um",              # 2*dx = separation distance of JJ's

    # GDS Export Option
    "GDS"            : False,           # True for GDS export, False for HFSS export

    # L_Turn Design
    "include_L"      : True,           # Set True to include the L_Turn, set False to remove it
    "L_offset"       : "-40um",
    "L_Turn_length"  : "120um",         # Length of the L Turn
    "L_spacing"      : "15um",           # Distance between the bottom of the qubit and the top of the L - wire
    "cpw_width_bias" : cpw_width_bias,  # Blue space of the bias wire
    "cpw_gap_bias"   : cpw_gap_bias,    # Gray space of the bias wire
    "flip"           : True,           # Flips (re-orients) the L-Turn 180 degrees
    "mirror"         : False,           # Mirrors the L-Turn
    "fillet"         : '99um'           # Fillet of L-turn (Radius Curvature)  -- default setting: 20um
}



t0 = make_qubit(t0_features, 'Q0_top', 'L_bracket0_top')

bias0_top = make_bias1('Qubit_0_Top_Control_Launch_Pad','tie', 'L_bracket0_top', 'second_end', 'bias0_top', start_straight='10um', end_straight='400um')



gui.rebuild()
gui.autoscale()

# Another Top Qubit

In [19]:

# Top 1 Qubit

t1_features = {
    # Position
    "pos_x"         : str( top_separation/2  ) + 'mm',               # x - position
    "pos_y"         : str(2.5 - top_vert_displacement ) + 'mm',   # y - position
    "orientation"   : "180",                                           # Angular Position

    # Qubit Features
    "pad_width"      : qubit_pad_width_t1,
    "pad_height"     : "0.1mm",
    "pocket_width"   : qubit_pocket_width_t1,       # Width of the gray space
    "pocket_height"  : "325um",         # Height of the gray space
    "big_gap"        : "60um",          # Separation space between the big capacitor and the edge of the pocket

    # Capacitor Features
    "include_a"      : True,           # Set True to include the rightmost mini-capacitor
    "include_b"      : True,            # Set True to include the leftmost mini-capacitor
    "mini_gap"       : "45um",          # Separation space between the mini capacitor and the big capacitor
    "mini_gap2"       : "45um",          # Separation space between the mini capacitor and the big capacitor  
    "mini_cap_length": 0.001,       # Length of the mini capacitor
    "mini_cap_lengthb": mini_cap_length_t1,       # Length of the mini capacitor   
    "outer_length"   : outer_length_t1,

    # Wire Features
    "cpw_width"      : cpw_width,       # Blue space of the wire
    "cpw_gap"        : cpw_gap,         # Gray space of the wire

    # Junctions
    "double"         : True,            # True for 2 x JJ's, False for 1 x JJ 
    "dx"             :"10um",              # 2*dx = separation distance of JJ's

    # GDS Export Option
    "GDS"            : False,           # True for GDS export, False for HFSS export

    # L_Turn Design
    "include_L"      : True,           # Set True to include the L_Turn, set False to remove it
    "L_offset"       : "40um", 
    "L_Turn_length"  : "120um",         # Length of the L Turn
    "L_spacing"      : "15um",           # Distance between the bottom of the qubit and the top of the L - wire
    "cpw_width_bias" : cpw_width_bias,  # Blue space of the bias wire
    "cpw_gap_bias"   : cpw_gap_bias,    # Gray space of the bias wire
    "flip"           : True,           # Flips (re-orients) the L-Turn 180 degrees
    "mirror"         : True,           # Mirrors the L-Turn
    "fillet"         : '99um'           # Fillet of L-turn (Radius Curvature)  -- default setting: 20um
}



t1 = make_qubit(t1_features, 'Q1_top', 'L_bracket1_top')
bias1_top = make_bias1('Qubit_1_Top_Control_Launch_Pad','tie', 'L_bracket1_top', 'second_end', 'bias1_top', start_straight='100um', end_straight='400um')

Q0_top_meander = make_meander('TQ0', 'second_end', 'Q0_top', 'a', L2_eff, 'meander_Q0_top', spacing = '100um', asymmetry= '300um',fillet = '40um')

Q1_top_meander = make_meander('TQ3', 'second_end', 'Q1_top', 'b', L1_eff, 'meander_Q1_top', spacing = '90um', asymmetry= '-150um',fillet = '40um')


gui.rebuild()
gui.autoscale()

# Qubit Coupler

In [20]:
ops=dict(fillet= '40um')

options_coupler = Dict(
    total_length= L2_coupled_eff , # Length of the 2 qubit  coupler (meander)
    hfss_wire_bonds = False,
    trace_width = cpw_width,
    trace_gap   = cpw_gap,

    meander = Dict(
        spacing = '90um',
        asymmetry = '-200um'
    ),

    pin_inputs=Dict(
        start_pin=Dict(
            component='Q0_top',
            pin='b'),
        end_pin=Dict(
            component='Q1_top',
            pin='a')),
    lead=Dict( # Only for meander use start & end straights
        start_straight= '0.10mm',
        end_straight  = '0.10mm'
        ),
    **ops
)

#two_qubit_coupler = RouteStraight(design, 'two_qubit_coupler', options=options_coupler)
two_qubit_coupler = RouteMeander(design, 'two_qubit_coupler', options=options_coupler)



gui.rebuild()
gui.autoscale()

# HFSS Setup

In [21]:
from qiskit_metal import Dict
from qiskit_metal import Dict
from qiskit_metal.analyses.quantization import EPRanalysis

eig_qb = EPRanalysis(design, "hfss")
q_hfss = eig_qb.sim.renderer

hfss_vars = Dict(
    Q0_top_Lj="6.01 nH",
    Q0_top_Cj="4 fF",

    Q1_top_Lj="6.87 nH",
    Q1_top_Cj="4 fF",

    Q0_bottom_Lj="11.6 nH",
    Q0_bottom_Cj="4 fF",
)

qcomps = design.components
qcomps['Q0_bottom'].options['hfss_inductance'] = 'Q0_bottom_Lj'
qcomps['Q0_bottom'].options['hfss_capacitance'] = 'Q0_bottom_Cj'
qcomps['Q0_top'].options['hfss_inductance'] = 'Q0_top_Lj'
qcomps['Q0_top'].options['hfss_capacitance'] = 'Q0_top_Cj'
qcomps['Q1_top'].options['hfss_inductance'] = 'Q1_top_Lj'
qcomps['Q1_top'].options['hfss_capacitance'] = 'Q1_top_Cj'
gui.rebuild()  # line needed to propagate the updates from the qubit instance into the junction design table
gui.autoscale()

q_hfss.start()

# Render only; this does not solve
rendered_design_name = q_hfss.execute_design(
    design_name="Full_Chip_Eigenmode",
    solution_type="eigenmode",
    vars_to_initialize=hfss_vars,
    force_redraw=True,
    selection=[],          
    open_pins=[],
    port_list=[],
    jj_to_port=[],
    ignored_jjs=[],
    box_plus_buffer=True,
)

print(rendered_design_name)

INFO 12:35PM [connect_project]: Connecting to Ansys Desktop API...
INFO 12:35PM [load_ansys_project]: 	Opened Ansys App
INFO 12:35PM [load_ansys_project]: 	Opened Ansys Desktop v2026.1.2
INFO 12:35PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/012738063/Documents/Ansoft/
	Project:   Project29
INFO 12:35PM [connect_design]: No active design found (or error getting active design).
INFO 12:35PM [connect]: 	 Connected to project "Project29". No design detected
INFO 12:35PM [connect_design]: 	Opened active design
	Design:    Full_Chip_Eigenmode [Solution type: Eigenmode]
WARNING 12:35PM [connect_setup]: 	No design setup detected.
WARNING 12:35PM [connect_setup]: 	Creating eigenmode default setup.
INFO 12:35PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


Full_Chip_Eigenmode


In [22]:
q_hfss.close()

True